# Задание №3. Прикладные дифференциальные уравнения

---

## 3.1. Задача о равномерном спуске

Найти кривую, по которой тяжёлая точка под действием силы тяжести **равномерно опускается** по вертикали с постоянной скоростью $v_y = -v_0 = \text{const}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

g = 9.81
v_y = -1.5
k = g / v_y**2

def y_curve(x, alpha):
    u = alpha**(-3) - 3 * k * x
    return (1 / (2 * k)) * (alpha**(-2) - np.cbrt(u**2))

x = np.linspace(-0.5, 0.0, 500)
alphas = [0.4, 0.8, 1.5]

plt.figure(figsize=(8, 6))
for a in alphas:
    y = y_curve(x, a)
    plt.plot(x, y, label=f'α = {a}')
plt.scatter([0], [0], color='blue')
plt.text(0, 0, ' Начало', va='bottom', ha='left')
plt.axhline(0, color='gray', lw=0.7)
plt.axvline(0, color='gray', lw=0.7)
plt.xlabel('x, м')
plt.ylabel('y, м')
plt.title('Кривые спуска')
plt.grid(True)
plt.legend()
plt.savefig('descent_static.png', dpi=250)
plt.show()

alpha = 0.8
def x_from_y(y):
    inner = alpha**(-2) - 2 * k * y
    return (alpha**(-3) - np.power(inner, 1.5)) / (3 * k)

x_curve = np.linspace(-0.5, 0.0, 500)
y_curve_vals = y_curve(x_curve, alpha)
y_start = 0.0
y_end = -0.3
T = (y_start - y_end) / abs(v_y)
t_vals = np.linspace(0, T, 250)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(x_curve, y_curve_vals, label='Траектория')
dot, = ax.plot([0], [0], 'bo', label='Точка')
ax.axhline(0, color='gray', lw=0.7)
ax.axvline(0, color='gray', lw=0.7)
ax.set_xlabel('x, м')
ax.set_ylabel('y, м')
ax.set_title('Анимация спуска')
ax.grid(True)
ax.legend(loc='lower left')
text_time = ax.text(0.02, 0.98, '', transform=ax.transAxes)

def init():
    dot.set_data([0], [0])
    text_time.set_text('')
    return dot, text_time

def update(frame):
    t = t_vals[frame]
    y = y_start + v_y * t
    x = x_from_y(y)
    dot.set_data([x], [y])
    text_time.set_text(f't = {t:.2f} с')
    return dot, text_time

anim = FuncAnimation(fig, update, frames=len(t_vals), init_func=init, interval=35, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

# Задача 3.2: Бильярд в прямоугольной области

### Теоретический анализ
Изучаем траекторию бильярдного шара в прямоугольнике со сторонами a и b, с упругими столкновениями от стенок.
Координаты: 0 ≤ x ≤ a, 0 ≤ y ≤ b. Начало: (x0, y0), скорость (vx, vy).
Внутри области движение прямолинейное: x(t) = x0 + vx t, y(t) = y0 + vy t.
При столкновении: для вертикальных стенок vx → -vx; для горизонтальных vy → -vy.
Метод развертывания: эквивалентно прямолинейному движению на плоскости с периодическим отображением обратно в прямоугольник.
Развернутые координаты: X(t) = x0 + vx t, Y(t) = y0 + vy t.
Сворачивание: x = fold(X mod 2a, a), y = fold(Y mod 2b, b), где fold(u, L) = u если u ≤ L, иначе 2L - u.
Тип траектории: если vx/vy рационально — периодическая; иррационально — эргодическая (плотно заполняет область).
Гамильтониан: H = (p_x^2 + p_y^2)/(2m), с сохранением импульсов внутри, знаки меняются при отражениях.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

a, b = 5, 3
x0, y0 = 0.8, 0.6
vx, vy = 1.0, 0.8
T = 50
N = 6000

t = np.linspace(0, T, N)
X = x0 + vx * t
Y = y0 + vy * t

def fold(u, L):
    u_mod = u % (2 * L)
    return np.where(u_mod <= L, u_mod, 2 * L - u_mod)

x = fold(X, a)
y = fold(Y, b)

plt.figure(figsize=(9, 5))
plt.plot(x, y, lw=0.8)
plt.xlim(0, a)
plt.ylim(0, b)
plt.grid(True)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Траектория бильярда')
plt.gca().set_aspect('equal')
plt.savefig('billiard_static.png', dpi=250)
plt.show()

N_anim = 700
t_anim = np.linspace(0, T, N_anim)
X_anim = x0 + vx * t_anim
Y_anim = y0 + vy * t_anim
x_anim = fold(X_anim, a)
y_anim = fold(Y_anim, b)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x_anim, y_anim, alpha=0.3)
dot, = ax.plot([], [], 'go', ms=7)
ax.set_xlim(0, a)
ax.set_ylim(0, b)
ax.set_aspect('equal')
ax.grid(True)
ax.set_title('Анимация бильярда')

def init():
    dot.set_data([], [])
    return dot,

def update(frame):
    dot.set_data([x_anim[frame]], [y_anim[frame]])
    return dot,

anim = FuncAnimation(fig, update, frames=N_anim, init_func=init, interval=15)
plt.close(fig)
HTML(anim.to_jshtml())

## 3.3. Цепная линия (катеноида)

$$ \frac{d}{ds} \left( T \frac{dy}{ds} \right) = \rho g $$

Решение: $ y = a \cosh(x/a) $

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

a = 1.2
C = 0.5
x = np.linspace(-6, 6, 500)
y = a * np.cosh(x / a) + C
plt.figure(figsize=(8, 5))
plt.plot(x, y, label=f'a = {a}')
plt.title('Цепная линия')
plt.xlabel('x')
plt.ylabel('y')
plt.grid(True)
plt.legend()
plt.savefig('catenary_static_single.png', dpi=250)
plt.show()

a_values = [0.6, 1.0, 2.5, 3.5]
plt.figure(figsize=(8, 5))
for a in a_values:
    y = a * np.cosh(x / a)
    plt.plot(x, y, label=f'a = {a}')
plt.title('Цепные линии для разных a')
plt.xlabel('x')
plt.ylabel('y')
plt.grid(True)
plt.legend()
plt.savefig('catenary_static_multiple.png', dpi=250)
plt.show()

a_anim = np.linspace(0.6, 3.5, 250)
fig, ax = plt.subplots(figsize=(8, 5))
line, = ax.plot([], [], lw=2.5)
ax.set_xlim(-6, 6)
ax.set_ylim(0, 12)
ax.grid(True)
ax.set_xlabel('x')
ax.set_ylabel('y')

def init():
    line.set_data([], [])
    return line,

def update(frame):
    a = a_anim[frame]
    y = a * np.cosh(x / a)
    line.set_data(x, y)
    ax.set_title(f'Цепная линия, a = {a:.2f}')
    return line,

anim = FuncAnimation(fig, update, frames=len(a_anim), init_func=init, interval=25, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

## 3.4. Таутохрона — циклоида

Решение уравнения Абеля: циклоида

$$ x = a(\theta - \sin \theta), \quad y = a(1 - \cos \theta) $$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

r = 1.2
t = np.linspace(0, 2*np.pi, 500)
x = r * (t - np.sin(t))
y = r * (1 - np.cos(t))
plt.figure(figsize=(8, 5))
plt.plot(x, y, label='Циклоида')
plt.title('Таутохронная кривая')
plt.xlabel('x')
plt.ylabel('y')
plt.grid(True)
plt.legend()
plt.gca().invert_yaxis()
plt.savefig('cycloid_static.png', dpi=250)
plt.show()

starts = [0.3*np.pi, 0.6*np.pi, 0.9*np.pi, 1.2*np.pi]
plt.figure(figsize=(8, 5))
for ts in starts:
    idx = t >= ts
    plt.plot(x[idx], y[idx], label=f'Старт φ={ts:.2f}')
plt.title('Траектории спуска')
plt.xlabel('x')
plt.ylabel('y')
plt.grid(True)
plt.legend()
plt.gca().invert_yaxis()
plt.savefig('cycloid_paths.png', dpi=250)
plt.show()

starts = [0.4*np.pi, 0.8*np.pi, 1.3*np.pi]
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x, y, color='silver', alpha=0.5)
points = [ax.plot([], [], 'mo')[0] for _ in starts]
ax.set_xlim(min(x), max(x))
ax.set_ylim(max(y), min(y))
ax.set_title('Анимация спуска')
ax.grid(True)

def init():
    for p in points:
        p.set_data([], [])
    return points

def update(frame):
    for i, ts in enumerate(starts):
        t_cur = ts + frame * 0.015
        if t_cur >= 2*np.pi:
            t_cur = 2*np.pi
        px = r * (t_cur - np.sin(t_cur))
        py = r * (1 - np.cos(t_cur))
        points[i].set_data([px], [py])
    return points

anim = FuncAnimation(fig, update, init_func=init, frames=350, interval=25, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

## 3.5. Математический маятник

$$ \ddot{\theta} + \frac{g}{l} \sin \theta = 0 $$

In [ ]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def pend_eqs(y, t, g, L):
    theta, omega = y
    return [omega, -(g / L) * np.sin(theta)]

def simulate_pendulum(theta0=0.0, omega0=0.0, T=12, dt=0.015, g=9.81, L=1.2):
    t = np.arange(0, T, dt)
    y0 = [theta0, omega0]
    sol = odeint(pend_eqs, y0, t, args=(g, L))
    return t, sol[:, 0], sol[:, 1]

t1, th1, _ = simulate_pendulum(np.deg2rad(15), 0.0)
t2, th2, _ = simulate_pendulum(np.deg2rad(50), 0.0)
t3, th3, _ = simulate_pendulum(np.deg2rad(85), 0.0)
plt.figure(figsize=(9, 5))
plt.plot(t1, np.rad2deg(th1), label='15°')
plt.plot(t2, np.rad2deg(th2), label='50°')
plt.plot(t3, np.rad2deg(th3), label='85°')
plt.xlabel('t, с')
plt.ylabel('θ, °')
plt.title('Колебания маятника')
plt.grid(True)
plt.legend()
plt.savefig('pendulum_angles.png', dpi=250)
plt.show()

g = 9.81
L = 1.2
t, theta, omega = simulate_pendulum(np.deg2rad(70), 0.0)
x = L * np.sin(theta)
y = -L * np.cos(theta)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))
ax1.set_xlim(-1.5*L, 1.5*L)
ax1.set_ylim(-1.5*L, 0.3*L)
ax1.set_aspect('equal')
ax1.grid(True)
ax1.set_title('Маятник')
rod, = ax1.plot([], [], '-o', lw=2.5)
ax2.set_xlim(-np.pi, np.pi)
ax2.set_ylim(-3.5, 3.5)
ax2.grid(True)
ax2.set_xlabel('θ, рад')
ax2.set_ylabel('ω, рад/с')
ax2.set_title('Фазовый портрет')
phase_traj, = ax2.plot([], [], alpha=0.4)
phase_point, = ax2.plot([], [], 'go')

def init():
    rod.set_data([], [])
    phase_traj.set_data([], [])
    phase_point.set_data([], [])
    return rod, phase_traj, phase_point

def update(frame):
    rod.set_data([0, x[frame]], [0, y[frame]])
    phase_traj.set_data(theta[:frame+1], omega[:frame+1])
    phase_point.set_data([theta[frame]], [omega[frame]])
    return rod, phase_traj, phase_point

anim = FuncAnimation(fig, update, init_func=init, frames=len(t), interval=25, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())